In [1]:
import sys
import random
import time
from datetime import datetime
from glob import glob
import warnings
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from pandas import DataFrame
from scipy.stats import genextreme, chi2
from scipy.optimize import minimize
from numpy import (ndarray, full_like, inf, exp, zeros, std, array, mean, 
                   min, max, sqrt, log, arange, percentile, ones, sum, linalg, random
                   )
from joblib import Parallel, delayed
from tqdm import tqdm
from collections import OrderedDict

import func_gev as gev
import func_preparation as dbf
import func_plotting as dbplt
import func_utils as ut

warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

# Settings

In [2]:
path_input = '../input/Annual_max_DCPP_20260112/'
path_export = '../output/gev_analysis/'

hindcast_start = 1960
hindcast_end = 2026

return_periods = [10, 25, 50, 100, 200]
plot_period_evolution = ['10-year', '50-year', '100-year']
t_eval = 2025

# adding a little randomness to enhance trust the model works as robust as possible cross locations..
start_location = random.choice(arange(0, 9579))
end_location = start_location+10
print(f'analyse a subsample of location {start_location}–{end_location}')


colors = [
    '#53354DFF','#7D4F73FF','#B887ADFF','#CAA5C2FF','#DBC3D6FF','#F5F5F5FF','#99E3DDFF',
    '#66D4CCFF','#33C6BBFF','#008A80FF','#005C55FF'
    ]

_LOCATION_LABELS = None
export_report=True
display_results = False

analyse a subsample of location 4670–4680


# Import Data

In [3]:
ls_files = [file for file in glob(path_input + '*.nc')]

print('Importing Data from ...')
print("\n".join(ls_files))

dic_data_per_model = dbf.import_all_models(ls_files)


Importing Data from ...
../input/Annual_max_DCPP_20260112/Annual_max_MIROC6.nc
../input/Annual_max_DCPP_20260112/Annual_max_MPI-ESM1-2-HR.nc
../input/Annual_max_DCPP_20260112/Annual_max_HadGEM3-GC31-MM.nc
../input/Annual_max_DCPP_20260112/Annual_max_MRI-ESM2-0.nc
../input/Annual_max_DCPP_20260112/Annual_max_BCC-CSM2-MR.nc
../input/Annual_max_DCPP_20260112/Annual_max_CMCC-CM2-SR5.nc
../input/Annual_max_DCPP_20260112/Annual_max_CanESM5.nc
../input/Annual_max_DCPP_20260112/Annual_max_NorCPM1.nc


# Prepare Data

### Pooling, BiasCorrection, ValidityCheck

In [4]:
print('Pooling and Preparing Data...')
dic_data_per_model, combined, notes_overview = dbf.prepare_combined_data(ls_files, dic_data_per_model)
print('... done.')

Pooling and Preparing Data...
... done.


### Rearrangement to per location

In [5]:
print('Rearranging Data – sorting per location...')    
dic_data_per_location = dbf.extract_location_data(combined, hindcast_start, hindcast_end)

Rearranging Data – sorting per location...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    6.4s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    6.4s
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:    6.6s
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:    6.7s
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    6.7s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.19251301043728633s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:    6.8s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:    6.9s
[Parallel(n_jobs=-1)]: Batch computation too fast (0.1591172218322754s.) Setting batch_size=4.
[Parallel(n_jobs=-1)]: Done  82 tasks      | elapsed:    7.1s
[Parallel(n_jobs=-1)]: Done 128 tasks      | elapsed:    7.3s
[Parallel(n_jobs=-1)]: Done 188 tasks      | elapsed:    7.5s
[Parallel(n_jobs=-1)]: Done 248 tasks      | elapsed:    7.8s
[Parallel(n_jobs=-1)]: Done 316 tasks      | elapse

## Select Subset

In [6]:
if start_location is not None or end_location is not None: 
    dic_data_per_location = ut.select_allowed_locations(
        dic_data_per_location=dic_data_per_location, 
        start_loc=start_location, end_loc=end_location
        )
    print(f'Processing locations {start_location} to {end_location} ({len(dic_data_per_location)} total)')

else:
    print(f'Processing all {len(dic_data_per_location)} locations')

print('Getting closest point available as location label for orientation. \nNote this is not the exact location...')
location_labels = dbf.precompute_location_labels(dic_data_per_location)
location_labels

Processing locations 4670 to 4680 (11 total)
Getting closest point available as location label for orientation. 
Note this is not the exact location...
Loading formatted geocoded file...


{(0.082367, 53.619732): 'Easington England GB',
 (0.083936, 35.950731): 'Mostaganem Mostaganem DZ',
 (0.095394, 38.668002): 'Moraira Valencia ES',
 (0.095764, 53.686265): 'Holmpton England GB',
 (0.095998, 53.497618): 'Tetney England GB',
 (0.109497, 53.611582): 'Easington England GB',
 (0.116648, 53.574064): 'Easington England GB',
 (0.118597, 53.493793): 'Tetney England GB',
 (0.123148, 53.595914): 'Easington England GB',
 (0.129758, 53.653106): 'Easington England GB',
 (0.133517, 40.055745): 'Orpesa/Oropesa del Mar Valencia ES'}

# (Non-)Stationary GEV Analysis with Pooled Data

When fitting a GEV to annual maxima, there are multiple sources of uncertainty in GEV analysis
- Parameter uncertainty (estimation uncertainty)
- Natural variability / return level uncertainty

**Parameter uncertainty**<br>
The GEV has parameters $(μ, σ, ξ)$ <br>
Each of these parameters is estimated from finite data, so each has an associated uncertainty.
This “uncertainty in the location parameter" can be captured by:
- Parametric sampling: sample μ ~ Normal(μ̂, SE_μ)
- Bootstrap: refit GEV on resampled data

**Natural variability / return level uncertainty** <br>
Even if parameters were known exactly, extreme values themselves are random.
The return level $z_T$ is defined as a high quantile of the GEV:
$z_T = μ + σ/ξ · [(-ln(1-1/T))^ξ - 1]$
<br>
When sampling from fitted GEV distribution, we can get a confidence interval for $z_T$ given fixed parameters.
This is the “return level uncertainty” captured by gev_return_levels_ci.

**Key Differences**
|Concept| How it's captured| Effect|
|---|---|---|
|μ (location) uncertainty	|Parametric bootstrap or μ-sampling from estimated SE	|Adds spread to the estimated parameter itself|
|Return level uncertainty	|Sampling from GEV with fixed parameters	|Adds spread due to natural variability of extremes|
|Combined uncertainty	|Sample μ from its distribution, then compute z_T from each μ	|Gives realistic CIs for return levels, including both sources|


In practice, best approach:
- Sample μ (location) from its uncertainty distribution
- For each μ, compute the return level $z_T$ using the quantile formula
- Compute percentile (median, 95% CI) → includes both parameter uncertainty and natural variability

In [7]:
def stationary_gev_incl_uncertainty(data, B=500, seed=None):
    """
    Estimate uncertainty of GEV location parameter using parametric bootstrap.
    
    Parameters
    ----------
    data : array
        Annual maxima
    B : int
        Number of bootstrap simulations
    seed : int or None
    
    Returns
    -------
    dict with:
        mu_hat
        mu_std
        mu_samples
    """
    if seed is not None:
        random.seed(seed)
    
    n = len(data)
    c_hat, mu_hat, sigma_hat = genextreme.fit(data)
    
    shape_samples = zeros(B)
    mu_samples = zeros(B)
    scale_samples = zeros(B)

    for b in range(B):
        synthetic = genextreme.rvs(
            c=c_hat,
            loc=mu_hat,
            scale=sigma_hat,
            size=n
        )
        
        c_b, mu_b, sigma_b = genextreme.fit(synthetic)  
        shape_samples[b] = -c_b
        mu_samples[b] = mu_b
        scale_samples[b] = sigma_b
        
    
    return {
        "mu_hat": mu_hat,
        "mu_std": std(mu_samples, ddof=1),
        "mu_samples": mu_samples,
        "shape_hat": -c_hat,
        "shape_std": std(shape_samples, ddof=1),
        "scale_hat": sigma_hat,
        "scale_std": std(scale_samples, ddof=1),
        "n_obs": n
    }


def bootstrap_stationary_gev_for_location(loc_id, data, B=300, seed=None):
    result = stationary_gev_incl_uncertainty(data, B=B, seed=seed)
    return loc_id, result

In [8]:
def fit_nonstationary_gev_with_uncertainty(
    years: ndarray, data: ndarray, trend_params: str = 'location', B: int = 500, seed=None
):
    """
    Fit non-stationary GEV where parameters vary linearly with time,
    and compute uncertainties using parametric bootstrap.

    Parameters
    ----------
    years : np.ndarray
        Years corresponding to each observation
    data : np.ndarray
        Annual maxima values
    trend_params : str
        Which parameters have trends: 'location', 'scale', or 'both'
    B : int
        Number of bootstrap samples
    seed : int or None
        Random seed

    Returns
    -------
    dict
        MLEs and parameter standard deviations
    """
    if seed is not None:
        random.seed(seed)
        
    n = len(data)
    t = (years - years.mean()) / years.std()  
    
    c_hat, mu_hat, sigma_hat = genextreme.fit(data)
    xi_hat = -c_hat

    def neg_loglik(params):
        if trend_params == 'location':
            mu0, mu1, sigma, xi = params
            mu_t = mu0 + mu1 * t
            sigma_t = full_like(t, sigma)
        elif trend_params == 'scale':
            mu, sigma0, sigma1, xi = params
            mu_t = full_like(t, mu)
            sigma_t = sigma0 + sigma1 * t
        elif trend_params == 'both':
            mu0, mu1, sigma0, sigma1, xi = params
            mu_t = mu0 + mu1 * t
            sigma_t = sigma0 + sigma1 * t
        else:
            raise ValueError("trend_params must be 'location', 'scale', or 'both'")
        
        if any(sigma_t <= 0):
            return inf
        
        z = (data - mu_t) / sigma_t
        if abs(xi) < 1e-10:  
            ll = -sum(log(sigma_t)) - sum(z) - sum(exp(-z))
        else:
            term = 1 + xi * z
            if any(term <= 0):
                return inf
            ll = -sum(log(sigma_t)) - sum((1 + 1/xi) * log(term)) - sum(term**(-1/xi))
        return -ll

    if trend_params == 'location':
        x0 = [mu_hat, 0.0, sigma_hat, xi_hat]
    elif trend_params == 'scale':
        x0 = [mu_hat, sigma_hat, 0.0, xi_hat]
    else:  
        x0 = [mu_hat, 0.0, sigma_hat, 0.0, xi_hat]

    result = minimize(neg_loglik, x0, method='Nelder-Mead')
    params_hat = result.x

    mu0_samples = []
    mu1_samples = []
    sigma_samples = []
    xi_samples = []

    for b in range(B):
        if trend_params == 'location':
            mu0, mu1, sigma, xi = params_hat
            mu_t = mu0 + mu1 * t
            synthetic = genextreme.rvs(c=-xi, loc=mu_t, scale=sigma, size=n)

            def neg_loglik_b(theta):
                mu0_b, mu1_b, sigma_b, xi_b = theta
                mu_t_b = mu0_b + mu1_b * t
                sigma_t_b = full_like(t, sigma_b)
                z = (synthetic - mu_t_b)/sigma_t_b
                term = 1 + xi_b*z
                if any(term <= 0) or any(sigma_t_b <=0):
                    return inf
                ll = -sum(log(sigma_t_b)) - sum((1+1/xi_b)*log(term)) - sum(term**(-1/xi_b))
                return -ll
            res_b = minimize(neg_loglik_b, params_hat, method='Nelder-Mead')
            mu0_b, mu1_b, sigma_b, xi_b = res_b.x
        
        mu0_samples.append(mu0_b)
        mu1_samples.append(mu1_b)
        sigma_samples.append(sigma_b)
        xi_samples.append(xi_b)

    mu0_samples = array(mu0_samples)
    mu1_samples = array(mu1_samples)
    sigma_samples = array(sigma_samples)
    xi_samples = array(xi_samples)
    
    return {
        'params_hat': params_hat,
        'mu0_hat': mu0_samples.mean(),
        'mu0_std': mu0_samples.std(ddof=1),
        'mu0_samples': mu0_samples,
        'mu1_hat': mu1_samples.mean(),
        'mu1_std': mu1_samples.std(ddof=1),
        'mu1_samples': mu1_samples,
        'sigma_hat': sigma_samples.mean(),
        'sigma_std': sigma_samples.std(ddof=1),
        'xi_hat': xi_samples.mean(),
        'xi_std': xi_samples.std(ddof=1),
        't_ref': years.mean(),
        't_std': years.std(),
        'n_obs': n
    }
    
    
def bootstrap_nonstationary_gev_for_location(loc_id, years, data, trend_params='location', B=300, seed=None):
    result = fit_nonstationary_gev_with_uncertainty(years, data, trend_params, B=B, seed=seed)
    return loc_id, result


In [9]:
def compute_return_levels_for_year(stationary, nonstationary, T, t_eval):
    # stationary
    mu = stationary['mu_hat']
    sd_mu = stationary['mu_std']
    z_T = mu + stationary['scale_hat']/stationary['shape_hat']*((-log(1-1/T))**(-stationary['shape_hat'])-1)
    z_lower = z_T - 1.96*sd_mu
    z_upper = z_T + 1.96*sd_mu
    

    # non-stationary
    t_scaled_eval = (t_eval - nonstationary['t_ref']) / nonstationary['t_std']
    mu_t = nonstationary['mu0_hat'] + nonstationary['mu1_hat']*t_scaled_eval
    mu_samples_eval = nonstationary['mu0_samples'] + nonstationary['mu1_samples']*t_scaled_eval
    sd_mu_t = std(mu_samples_eval, ddof=1)
    z_T_ns = mu_t + nonstationary['sigma_hat']/nonstationary['xi_hat']*((-log(1-1/T))**(-nonstationary['xi_hat'])-1)
    z_lower_ns = z_T_ns - 1.96*sd_mu_t
    z_upper_ns = z_T_ns + 1.96*sd_mu_t
    
    return dict({
        'stationary': {'z_T': z_T, 'lower': z_lower, 'upper': z_upper},
        'nonstationary': {'z_T': z_T_ns, 'lower': z_lower_ns, 'upper': z_upper_ns},
            })


In [10]:
def likelihood_ratio_test(LL_s, LL_ns, df):
    delta_LL = 2 * (LL_ns - LL_s)
    if delta_LL < 0:
        return delta_LL, 1.0, '→ stationary model is sufficient, trend in μ is not significant'
    else:
        p = 1 - chi2.cdf(delta_LL, df)
        if p< 0.05:
            return delta_LL, p, '→ non-stationary model is significantly better → μ(t) trend matters'
        else:
            return delta_LL, p, '→ adding a trend doesn’t improve the fit'


def compare_stationary_nonstationary(stationary, nonstationary, data_loc):
    n = stationary['n_obs']

    # Stationary
    mu_s = stationary['mu_hat']
    sigma_s = stationary['scale_hat']
    xi_s = stationary['shape_hat']

    z_s = (data_loc.annual_max - mu_s)/sigma_s
    if abs(xi_s) < 1e-10:
        LL_s = -sum(log(sigma_s)) - sum(z_s) - sum(exp(-z_s))
    else:
        term = 1 + xi_s*z_s
        LL_s = -sum(log(sigma_s)) - sum((1+1/xi_s)*log(term)) - sum(term**(-1/xi_s))
    k_s = 3  # stationary: μ, σ, ξ
    AIC_s = 2*k_s - 2*LL_s
    BIC_s = k_s*log(n) - 2*LL_s


    # Non-stationary (location trend)
    mu0 = nonstationary['mu0_hat']
    mu1 = nonstationary['mu1_hat']
    sigma_ns = nonstationary['sigma_hat']
    xi_ns = nonstationary['xi_hat']
    t_scaled = (data_loc.year - nonstationary['t_ref']) / nonstationary['t_std']  # scaled years

    mu_t = mu0 + mu1*t_scaled
    z_ns = (data_loc.annual_max - mu_t)/sigma_ns
    if abs(xi_ns) < 1e-10:
        LL_ns = -sum(log(sigma_ns)) - sum(z_ns) - sum(exp(-z_ns))
    else:
        term = 1 + xi_ns*z_ns
        LL_ns = -sum(log(sigma_ns)) - sum((1+1/xi_ns)*log(term)) - sum(term**(-1/xi_ns))

    k_ns = 4  # μ0, μ1, σ, ξ
    AIC_ns = 2*k_ns - 2*LL_ns
    BIC_ns = k_ns*log(n) - 2*LL_ns

    # Likelihood ratio test
    df = k_ns - k_s
    delta_LL, p_value, interpretation = likelihood_ratio_test(LL_s=LL_s, LL_ns=LL_ns, df=df)
    
    return dict({
            'stationary': {'LL': LL_s, 'k': k_s, 'AIC': AIC_s, 'BIC': BIC_s},
            'nonstationary': {'LL': LL_ns, 'k': k_ns, 'AIC': AIC_ns, 'BIC': BIC_ns},
            'LRT': {'delta_LL': delta_LL, 'df': df, 'p_value': p_value, 'interpretation': interpretation}
        })
    

In [11]:
def set_location_labels(labels):
    global _LOCATION_LABELS
    _LOCATION_LABELS = labels

In [12]:
def process_location(loc_id, dic_data_per_location, return_periods, t_eval, B=150, seed=None):
    """
    Fit stationary and non-stationary GEV, compute return levels and model comparison for one location.
    """
    results_loc = {}

    df_prepared = dic_data_per_location[loc_id]
    lon_loc = df_prepared.lon.unique()[0]
    lat_loc = df_prepared.lat.unique()[0]

    location_info = _LOCATION_LABELS.get((round(lon_loc,6), round(lat_loc,6)), "unknown location")
    print(f'GEV analysis for location {loc_id} · {location_info}')
    
    annual_max = gev.extract_annual_maxima_at_location(df_prepared, lon=lon_loc, lat=lat_loc)
    years = annual_max['year'].values
    data = annual_max['annual_max'].values

    if len(annual_max) < 10:
        return loc_id, {
            'location_info': location_info,
            'LatLon': (lat_loc, lon_loc),
            'error': f'Not enough data ({len(annual_max)})'
        }

    # ------------------ stationary GEV ------------------
    gev_stationary = stationary_gev_incl_uncertainty(data, B=B, seed=seed)

    # store original data for likelihood calculation
    gev_stationary['data'] = data

    # ------------------ non-stationary GEV ------------------
    gev_nonstat = fit_nonstationary_gev_with_uncertainty(years, data, trend_params='location', B=B, seed=seed)

    # store original data for non-stationary
    gev_nonstat['data'] = {'values': data, 'years': years}

    # ------------------ model comparison ------------------
    comparison = compare_stationary_nonstationary(gev_stationary, gev_nonstat, annual_max)

    # ------------------ return levels ------------------
    all_return_levels = {}
    for T in return_periods:
        rl = compute_return_levels_for_year(gev_stationary, gev_nonstat, T=T, t_eval=t_eval)
        all_return_levels[T] = rl

    # ------------------ store results ------------------
    results_loc = {
        'location_info': location_info,
        'LatLon': (lat_loc, lon_loc),
        'data': annual_max,
        'stationary': gev_stationary,
        'nonstationary': gev_nonstat,
        'model_comparison': comparison,
        'return_levels': all_return_levels
    }

    return loc_id, results_loc


## Compute for 1 Sample

In [ ]:
## UPDATE - testing for 1 location | later to be parallelized
# include uncertainty in stationary GEV parameters
B = 300 # bootstrap number - downscaling to 50?! (from 300)
seed = None

# ----------------------------------------------------------------------------------------------------------
results = dict()
loc_id = list(dic_data_per_location.keys())[0]
df_prepared = dic_data_per_location[loc_id]

lon_loc = df_prepared.lon.unique()[0]
lat_loc = df_prepared.lat.unique()[0]

set_location_labels(location_labels)
location_info = _LOCATION_LABELS.get((round(lon_loc,6), round(lat_loc,6)), "unknown location")
print(f'GEV analysis for location {loc_id} · {location_info}')

# ----------------------------------------------------------------------------------------------------------
ls_notes = []
annual_max = gev.extract_annual_maxima_at_location(df_prepared, lon=lon_loc, lat=lat_loc)

if len(annual_max) < 10:
    print('WARNING - not enough data (<10) for location {loc_id} (lon|lat · {lon_loc}|{lat_loc})')

years = annual_max['year'].values
data = annual_max['annual_max'].values

print("\tConducting stationary GEV...")
gev_stationary = stationary_gev_incl_uncertainty(data, B=B, seed=seed)
print(f"\t → Stationary GEV done (success {gev_stationary != None})")

print("\tContinuing with non-stationary GEV...")
gev_nonstat = fit_nonstationary_gev_with_uncertainty(years, data, trend_params='location', B=B, seed=seed)
print(f"\t → Non-stationary GEV done (success {gev_nonstat != None})")

print("\tCompare Models...")
comparison = compare_stationary_nonstationary(gev_stationary, gev_nonstat, annual_max)

print("\tCompute Return Levels...")
all_return_levels = dict()
for T in return_periods:
    rl = compute_return_levels_for_year(gev_stationary, gev_nonstat, T=T, t_eval=t_eval)
    all_return_levels[T] = rl


results[loc_id] = dict({
    'location_info': location_info,
    'LatLon': (lat_loc, lon_loc),
    'data':annual_max, 
    'stationary': gev_stationary,
    'nonstationary': gev_nonstat,
    'model_comparison': comparison,
    'return_levels': all_return_levels
    })

GEV analysis for location 4670 · Easington England GB
	Conducting stationary GEV...
	 → Stationary GEV done (success True)
	Continuing with non-stationary GEV...
	 → Non-stationary GEV done (success True)
	Compare Models...
	Compute Return Levels...


In [18]:
loc_ex = list(results.keys())[0]

print(f'GEV Analysis Overview for location ID: {loc_ex}')

print('\nStationary GEV analysis')
gev_stat = results[loc_ex]['stationary']
print('shape:\t', gev_stat['shape_hat'].round(3), '±', gev_stat['shape_std'].round(3))
print('scale:\t', gev_stat['scale_hat'].round(3), '±', gev_stat['scale_std'].round(3))
print('µ:\t', round(gev_stat['mu_hat']*1000,2), '±', round(gev_stat['mu_std']*1000,2))


print('\nNON-Stationary GEV analysis')
gev_nonstat = results[loc_ex]['nonstationary']
print('MLE results:'
    '\t', 
    'µ0', gev_nonstat['params_hat'][0]*1000, '|', 
    'µ1', gev_nonstat['params_hat'][1]*1000, '|', 
    'scale', gev_nonstat['params_hat'][2], '|', 
    'shape', gev_nonstat['params_hat'][3]
    )
print('shape:\t', gev_nonstat['xi_hat'].round(3), '±', gev_nonstat['xi_std'].round(3))
print('scale:\t', gev_nonstat['sigma_hat'].round(3), '±', gev_nonstat['sigma_std'].round(3))
print('µ0:\t', round(gev_nonstat['mu0_hat']*1000,2), '±', round(gev_nonstat['mu0_std']*1000,2))
print('µ1:\t', round(gev_nonstat['mu1_hat']*1000,2), '±', round(gev_nonstat['mu1_std']*1000,2))

print('\nMODEL COMPARISON')
print(results[loc_ex]['model_comparison']['LRT']['interpretation'])
print('p_value', results[loc_ex]['model_comparison']['LRT']['p_value'])
print('delta_LL', results[loc_ex]['model_comparison']['LRT']['delta_LL'].round(3))


GEV Analysis Overview for location ID: 4670

Stationary GEV analysis
shape:	 -0.112 ± 0.008
scale:	 0.22 ± 0.003
µ:	 1072.02 ± 3.44

NON-Stationary GEV analysis
MLE results:	 µ0 1072.1411159601487 | µ1 -3.151515802483913 | scale 0.2197710287438739 | shape -0.11204389632564318
shape:	 -0.112 ± 0.009
scale:	 0.22 ± 0.002
µ0:	 1072.11 ± 3.69
µ1:	 -3.6 ± 3.08

MODEL COMPARISON
→ non-stationary model is significantly better → μ(t) trend matters
p_value 0.0020459710520809304
delta_LL 9.508


- location (µ) → central tendency
- shape (ξ) → linked to distribution (dist_type) 
- scale (σ) → spread

All three parameters are constant because it’s stationary: no dependence on time or covariates.

- µ0, µ1 → The location parameter is now time-dependent:
$μ(t) = μ_0 + μ_1 ⋅(t−t_{ref})$
- ξ (xi) → shape parameter constant across time
- σ (sigma) → scale parameter constant across time

## Compute For All (selected) Locations

In [16]:
return_periods, t_eval

([10, 25, 50, 100, 200], 2025)

In [ ]:
B = 300 # bootstrap number - downscaling to 50?! (from 300 ~ 3min20)
seed = None

bootstrap_results = Parallel(n_jobs=-1)(
    delayed(process_location)(
        loc_id,
        dic_data_per_location,
        return_periods,
        t_eval,
        B=B,
        seed=seed
    )
    for loc_id in tqdm(dic_data_per_location.keys())
)

results_all = dict(bootstrap_results)

100%|██████████| 11/11 [00:06<00:00,  1.82it/s]


GEV analysis for location 4672 · Moraira Valencia ES
GEV analysis for location 4670 · Easington England GB
GEV analysis for location 4671 · Mostaganem Mostaganem DZ
GEV analysis for location 4673 · Holmpton England GB
GEV analysis for location 4674 · Tetney England GB
GEV analysis for location 4675 · Easington England GB
GEV analysis for location 4676 · Easington England GB
GEV analysis for location 4677 · Tetney England GB
GEV analysis for location 4678 · Easington England GB
GEV analysis for location 4679 · Easington England GB
GEV analysis for location 4680 · Orpesa/Oropesa del Mar Valencia ES


# Annnual Stationary GEV Analysis

In [330]:
def annual_stationary_gev(loc_data, B=300, seed=None):
    """
    Compute stationary GEV per year at a single location.
    
    Parameters
    ----------
    loc_data : pd.DataFrame
        Must contain 'year' and 'annual_max' (or daily maxima) columns
    B : int
        Number of bootstrap simulations for parameter uncertainty
    seed : int or None
    
    Returns
    -------
    dict : keyed by year
        {'mu_hat', 'sigma_hat', 'xi_hat', 'mu_std', 'sigma_std', 'xi_std'}
    """
    
    results_yearly = OrderedDict()
    years = sorted(loc_data['year'].unique())

    for year in years:
        data_year = loc_data.loc[loc_data['year']==year, 'annual_max'].values
        if len(data_year) < 3:  # need at least 3 points to fit GEV
            results_yearly[year] = {'error': f'Not enough data ({len(data_year)})'}
            continue
        
        gev_result = stationary_gev_incl_uncertainty(data_year, B=B, seed=seed)
        results_yearly[year] = gev_result
    
    return results_yearly
